# Lab 8 solution: Find Groups, and Catch It When It Goes Wrong
All code as in the worksheet, run from top to bottom, with one possible answer after each Part.

# Lab 8: Find Groups, and Catch It When It Goes Wrong

## MPS311/439 Machine Learning

## What this lab is for

## How to work

## Part 0: Set up (3 minutes)

In [1]:
import numpy as np                                        # numbers and arrays
import pandas as pd                                       # tables
import matplotlib.pyplot as plt                           # plots
import time                                               # a clock, to time the fitting
import urllib.request                                     # to download a file
from sklearn.datasets import load_wine                    # the first dataset
from sklearn.model_selection import train_test_split, KFold   # splitting the data
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.decomposition import PCA                     # fewer numbers per image
from sklearn.cluster import KMeans, AgglomerativeClustering   # today's tools

## Part 1: Find groups in a table of wines (9 minutes)

In [2]:
data = load_wine()                                        # ← change: which dataset
X = data.data                                             # the 13 measurements (features)
y = data.target                                           # ← change: the labels (here the grape variety, 0, 1 or 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

(133, 13) [44 53 36]


In [3]:
km = KMeans(n_clusters=3, n_init=10, random_state=0)      # 3 groups, the best of 10 starting points
groups = km.fit_predict(X_train)                          # ← change: X_train or X_train_s; no labels are passed in
table = np.zeros((3, 3), dtype=int)                       # rows: groups, columns: grape varieties
for i in range(len(groups)):                              # every training wine
    table[groups[i], y_train[i]] += 1                     # add one to its group and its variety
print(table)                                              # the count table
print(round(table.max(axis=1).sum() / len(groups), 3))    # majority vote: each group guesses its most common variety
spread = pd.Series(X_train.std(axis=0), index=data.feature_names)   # the spread of each column
print(spread.sort_values(ascending=False).head(3).round(1))          # the three columns with the largest spread

[[ 0 38 14]
 [37  1  0]
 [ 7 14 22]]
0.729
proline              323.6
magnesium             13.8
alcalinity_of_ash      3.5
dtype: float64


The same cell with `X_train_s`:

In [4]:
km = KMeans(n_clusters=3, n_init=10, random_state=0)      # 3 groups, the best of 10 starting points
groups = km.fit_predict(X_train_s)                         # ← change: X_train or X_train_s; no labels are passed in
table = np.zeros((3, 3), dtype=int)                       # rows: groups, columns: grape varieties
for i in range(len(groups)):                              # every training wine
    table[groups[i], y_train[i]] += 1                     # add one to its group and its variety
print(table)                                              # the count table
print(round(table.max(axis=1).sum() / len(groups), 3))    # majority vote: each group guesses its most common variety
spread = pd.Series(X_train.std(axis=0), index=data.feature_names)   # the spread of each column
print(spread.sort_values(ascending=False).head(3).round(1))          # the three columns with the largest spread

[[ 0  4 36]
 [43  1  0]
 [ 1 48  0]]
0.955
proline              323.6
magnesium             13.8
alcalinity_of_ash      3.5
dtype: float64


**One possible answer.** `X_train`: 0.729. `X_train_s`: 0.955. Proline is measured in hundreds to over a thousand, with a spread of 323.6; the next largest spreads are 13.8 (magnesium) and 3.5. A distance between two wines is almost only the difference in proline, so K-means groups the wines by proline alone and ignores the other 12 measurements. After standardising, every column has spread 1 and all 13 count. Distances depend on the units.

**Going further, Part 1.** Within-cluster sums of squares for 1 to 8 groups: 1729.0, 1237.5, 960.1, 877.3, 822.0, 763.5, 730.2, 691.3. The falls are 491.5, 277.4, then 82.8: the curve bends at 3, but not sharply. On the test wines the guesses are right for 0.978 (44 of 45).

## Part 2: Sort the digits, as Assignment 2 asks (11 minutes)

In [5]:
DATA_URL = "https://raw.githubusercontent.com/wayXing/mps311-439-course-materials/main/lecture-slids/lessons/data/mnist10k.npz"
urllib.request.urlretrieve(DATA_URL, "mnist10k.npz")      # 10,000 handwritten digits, about 1.6 MB (if this fails, see the Appendix)
data = np.load("mnist10k.npz")                            # ← change: which dataset
X = data["X"] / 255                                       # the 784 pixels of each 28 x 28 image, scaled to 0 to 1
y = data["y"]                                             # ← change: the labels (the digit, 0 to 9)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column (not used for the digits today)
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

(7500, 784) [743 851 746 762 717 680 770 780 710 741]


In [6]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0, stratify=y_train)   # fit on 3/4, check on 1/4
print(X_fit.shape, X_val.shape, X_test.shape)             # fitting, validation, test

(5625, 784) (1875, 784) (2500, 784)


In [7]:
k = 10                                                    # ← change: the number of groups
pca = PCA(n_components=50, random_state=0)                # 50 components, as in Assignment 2
pca.fit(X_fit)                                            # learn the components from the fitting images only
Z_fit = pca.transform(X_fit)                              # 50 numbers for each fitting image
Z_val = pca.transform(X_val)                              # the same directions for the validation images
km = KMeans(n_clusters=k, n_init=10, random_state=0)      # k groups, the best of 10 starting points
km.fit(Z_fit)                                             # no labels are passed in
groups = km.labels_                                       # the group of every fitting image
print(round(km.inertia_, 1))                              # within-cluster sum of squares

169025.6


In [8]:
table = np.zeros((k, 10), dtype=int)                      # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print(table)                                              # the count table
print("guesses:", guess)                                  # one guess per group
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

[[  1 285  38   8  28  86  25  25  39  10]
 [ 32   1  22 334   2 168   2   2 106  13]
 [  4   0 413  10   7   1  48   4  10   7]
 [234   0   1   2   0   7   7   2   2   2]
 [251   0   2   5   0   8   6   2   3   2]
 [  1   0  11   6 169  30   0 332  28 222]
 [ 18   0  19 135   0 146   9   0 287   6]
 [  1   0  15  22 309  41   8 190  24 267]
 [  0 352  25  43  11  14  30  28  28  26]
 [ 15   0  14   6  12   9 443   0   5   1]]
guesses: [1 3 2 0 0 7 8 4 1 6]
fitting: 0.576
validation: 0.577


**One possible answer.** 0.576 on the fitting images and 0.577 on the validation images: the guesses work as well on images K-means has not seen. The guesses contain 0 twice and 1 twice, and no 5 or 9. With only 10 groups, 0 and 1 each took two (written slanted or upright), so there were no groups left for 5 and 9: most 5s sit in the groups that guess 3 and 8, and most 9s in the groups that guess 4 and 7. K-means makes compact groups; it does not know there should be one group per digit.

In [9]:
for seed in range(5):                                     # five different starting points
    km1 = KMeans(n_clusters=10, n_init=1, random_state=seed)   # only one start
    km1.fit(Z_fit)                                        # run K-means from that start
    t = np.zeros((10, 10), dtype=int)                     # count table for this run
    for i in range(len(Z_fit)):                           # every fitting image
        t[km1.labels_[i], y_fit[i]] += 1                  # add one to its group and its digit
    print(seed, round(km1.inertia_, 1), round(t.max(axis=1).sum() / len(Z_fit), 3))   # seed, within-cluster sum of squares, majority vote

0 169033.3 0.586
1 169714.5 0.599
2 169349.4 0.594
3 169025.8 0.576
4 169026.9 0.581


**Going further, Part 2.** With 15 groups the votes are 0.673 (fitting) and 0.674 (validation), and every digit is some group's guess. More groups always give a higher vote: with 5625 groups, one image each, it would be 1.0. The vote rewards small pure groups, so it cannot choose k; k = 10 comes from knowing there are ten digits. Single starts: `0 169033.3 0.586`, `1 169714.5 0.599`, `2 169349.4 0.594`, `3 169025.8 0.576`, `4 169026.9 0.581` (the last digits may differ on another computer). The smallest sum has the lowest vote: K-means makes the groups compact, which is not the same as matching the digits.

## Part 3: Too many images for hierarchical clustering (7 minutes)

In [10]:
n = 2000                                                  # ← change: how many fitting images to use
rng = np.random.default_rng(0)                            # fixed random numbers, so everyone picks the same rows
rows = rng.choice(len(Z_fit), n, replace=False)           # n different rows, picked at random
Z_part = Z_fit[rows]                                      # their 50 numbers
y_part = y_fit[rows]                                      # their digits (only to check)
print("memory for every pair:", round(n * (n - 1) / 2 * 8 / 1e9, 3), "GB")   # n(n-1)/2 pairs, 8 bytes each
start = time.time()                                       # the clock before fitting
hc = AgglomerativeClustering(n_clusters=10, linkage="ward")   # 10 groups; Ward: merge where the spread grows least
groups_hc = hc.fit_predict(Z_part)                        # one group per image
print("seconds:", round(time.time() - start, 1))          # how long it took
table_hc = np.zeros((10, 10), dtype=int)                  # rows: groups, columns: true digits
for i in range(n):                                        # every image in the part
    table_hc[groups_hc[i], y_part[i]] += 1                # add one to its group and its digit
print("majority vote:", round(table_hc.max(axis=1).sum() / n, 3))   # each group guesses its most common digit

memory for every pair: 0.016 GB


seconds: 0.1
majority vote: 0.62


The same cell with `n = 500`:

In [11]:
n = 500                                                   # ← change: how many fitting images to use
rng = np.random.default_rng(0)                            # fixed random numbers, so everyone picks the same rows
rows = rng.choice(len(Z_fit), n, replace=False)           # n different rows, picked at random
Z_part = Z_fit[rows]                                      # their 50 numbers
y_part = y_fit[rows]                                      # their digits (only to check)
print("memory for every pair:", round(n * (n - 1) / 2 * 8 / 1e9, 3), "GB")   # n(n-1)/2 pairs, 8 bytes each
start = time.time()                                       # the clock before fitting
hc = AgglomerativeClustering(n_clusters=10, linkage="ward")   # 10 groups; Ward: merge where the spread grows least
groups_hc = hc.fit_predict(Z_part)                        # one group per image
print("seconds:", round(time.time() - start, 1))          # how long it took
table_hc = np.zeros((10, 10), dtype=int)                  # rows: groups, columns: true digits
for i in range(n):                                        # every image in the part
    table_hc[groups_hc[i], y_part[i]] += 1                # add one to its group and its digit
print("majority vote:", round(table_hc.max(axis=1).sum() / n, 3))   # each group guesses its most common digit

memory for every pair: 0.001 GB
seconds: 0.0
majority vote: 0.592


The same cell with `n = 4000`:

In [12]:
n = 4000                                                  # ← change: how many fitting images to use
rng = np.random.default_rng(0)                            # fixed random numbers, so everyone picks the same rows
rows = rng.choice(len(Z_fit), n, replace=False)           # n different rows, picked at random
Z_part = Z_fit[rows]                                      # their 50 numbers
y_part = y_fit[rows]                                      # their digits (only to check)
print("memory for every pair:", round(n * (n - 1) / 2 * 8 / 1e9, 3), "GB")   # n(n-1)/2 pairs, 8 bytes each
start = time.time()                                       # the clock before fitting
hc = AgglomerativeClustering(n_clusters=10, linkage="ward")   # 10 groups; Ward: merge where the spread grows least
groups_hc = hc.fit_predict(Z_part)                        # one group per image
print("seconds:", round(time.time() - start, 1))          # how long it took
table_hc = np.zeros((10, 10), dtype=int)                  # rows: groups, columns: true digits
for i in range(n):                                        # every image in the part
    table_hc[groups_hc[i], y_part[i]] += 1                # add one to its group and its digit
print("majority vote:", round(table_hc.max(axis=1).sum() / n, 3))   # each group guesses its most common digit

memory for every pair: 0.064 GB


seconds: 0.3
majority vote: 0.664


**One possible answer.** Majority votes: 0.592 (500), 0.620 (2000), 0.664 (4000), all at least as good as K-means on all 5625 images (0.576). Memory: 0.001, 0.016 and 0.064 GB; for 50,000 images 50,000 × 49,999 / 2 × 8 bytes ≈ 10 GB, more than Colab gives you. Doubling the images makes the memory four times larger. The votes differ because each `n` picks different images, and a few hundredths either way is no evidence that fewer or more images are better. In the assignment I would run hierarchical clustering on a random 5,000 images (about 0.1 GB) and K-means, which only needs the distance from each image to k centres, on all of them.

**Going further, Part 3.** Single: majority vote 0.119, and one group holds 1991 of the 2000 images. Images of different digits are linked through a chain of unclear images, as in the lecture.

## Part 4: Check the code an AI wrote (12 minutes)

In [13]:
km = KMeans(n_clusters=10, n_init=10, random_state=0)     # 10 groups, as in Part 2
km.fit(Z_fit)                                             # no labels are passed in
groups = km.labels_                                       # the group of every fitting image
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

fitting: 0.576
validation: 0.577


In [14]:
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((groups_val == y_val).mean(), 3))         # share whose group is their digit

fitting: 0.576


validation: 0.117


In [15]:
table = np.zeros((10, 10), dtype=int)                     # rows: groups, columns: true digits
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(Z_val)   # groups for the validation images
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

fitting: 0.576
validation: 0.192


In [16]:
table = np.zeros((10, 10), dtype=int)                     # a count table
for i in range(len(groups)):                              # every fitting image
    table[y_fit[i], groups[i]] += 1                       # add one to its digit and its group
guess = table.argmax(axis=1)                              # the most common entry of each row
print("fitting:", round(table.max(axis=1).sum() / len(groups), 3))   # majority vote on the fitting images
groups_val = km.predict(Z_val)                            # each validation image goes to its nearest centre
print("validation:", round((guess[groups_val] == y_val).mean(), 3))  # share whose digit is their group's guess

fitting: 0.561
validation: 0.106


**One possible answer.** - **A, validation 0.117: wrong.** It compares the group number with the digit. Group numbers are arbitrary: K-means might call the group of zeros "group 7". A group number only means something through the count table, as `guess[groups_val]`.
- **B, validation 0.192: wrong.** It runs K-means again on the validation images. That makes a new set of groups with new numbers, which have nothing to do with the guesses learned on the fitting images. Validation images must go to the centres learned from the fitting images: `km.predict`.
- **C, fitting 0.561, validation 0.106: wrong, although the fitting score looks normal.** The table has rows and columns swapped: each row is now a digit, and `max(axis=1)` counts, for each digit, its most common group. Two digits can claim the same group, so 0.561 is not the share of images guessed right. The guesses are now groups, not digits, so the validation score falls apart. Only the validation score gave it away.

**Going further, Part 4.** Typical one-line mistakes: PCA fitted on `X_train` or on all images (the scores hardly change, but the validation and test images are no longer unseen), `pca.fit_transform(X_val)` (new directions: the validation images land far from the centres), or the guesses taken from the validation labels. Several give normal-looking scores, which is why reading the code matters.

## Part 5: The test set, once (1 minute)

In [17]:
table = np.zeros((10, 10), dtype=int)                     # the correct count table again (Part 4 may have changed it)
for i in range(len(groups)):                              # every fitting image
    table[groups[i], y_fit[i]] += 1                       # add one to its group and its digit
guess = table.argmax(axis=1)                              # each group guesses its most common digit
groups_test = km.predict(pca.transform(X_test))           # the test images: transform and predict only
print(round((guess[groups_test] == y_test).mean(), 3))    # the 2,500 test images, read once

0.577


# After class

## How today's lab helps with Assignment 2

## Reflection